## Lab 6: RNN

### Objective:

To design and implement a Recurrent Neural Network (RNN) in PyTorch for learning and predicting patterns from sequential data.

### Theory:

Recurrent Neural Networks (RNNs) are a class of neural networks developed to handle data that appears in a sequence. Unlike conventional neural networks that process each input independently, RNNs maintain information from previous inputs through an internal memory called the hidden state. This capability allows them to capture temporal relationships and dependencies within sequential data.

RNNs are particularly useful when the order of information is important. Applications include natural language processing, speech recognition, machine translation, handwriting recognition, and time-series forecasting.

At each time step, the network receives the current input and combines it with information stored from the previous step. This combined information is passed through an activation function to generate an updated hidden state. The hidden state serves as a compact representation of past information and is continuously updated as new inputs arrive.

Mathematically, the hidden state can be represented as:
ht=tanh(Wih xt+Whh ht−1 +b)

In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================================
# 1. Generate dataset
# =====================================================

dishes = ['A', 'B', 'C']
weather_types = ['Sunny', 'Rainy']

dish_to_idx = {d: i for i, d in enumerate(dishes)}
weather_to_idx = {w: i for i, w in enumerate(weather_types)}

def next_dish(dish):
    if dish == 'A':
        return 'B'
    elif dish == 'B':
        return 'C'
    else:
        return 'A'

def generate_sequence(length=1000):
    """
    Returns:
        inputs  = [(dish_t, weather_t), ...]
        targets = [dish_{t+1}, ...]
    """
    current_dish = 'A'

    inputs = []
    targets = []

    for _ in range(length):

        weather = random.choice(weather_types)

        inputs.append((current_dish, weather))

        if weather == 'Sunny':
            new_dish = current_dish
        else:  # Rainy
            new_dish = next_dish(current_dish)

        targets.append(new_dish)

        current_dish = new_dish

    return inputs, targets

In [2]:
# =====================================================
# 2. Encode data
# =====================================================

def encode_input(dish, weather):
    """
    One-hot encode dish (3 dims)
    +
    One-hot encode weather (2 dims)

    Total input size = 5
    """
    x = torch.zeros(5)

    x[dish_to_idx[dish]] = 1.0
    x[3 + weather_to_idx[weather]] = 1.0

    return x


inputs, targets = generate_sequence(length=2000)

X = torch.stack([
    encode_input(d, w)
    for d, w in inputs
])

y = torch.tensor([
    dish_to_idx[t]
    for t in targets
])

# RNN expects:
# (batch_size, seq_len, input_size)

X = X.unsqueeze(0)      # (1, seq_len, 5)
y = y.unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([1, 2000, 5])
y shape: torch.Size([1, 2000])


In [4]:
# =====================================================
# 3. Vanilla RNN model
# =====================================================

class DishRNN(nn.Module):
    def __init__(self,
                 input_size=5,
                 hidden_size=3,
                 output_size=3):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True, 
            bias=False, 
            nonlinearity = "tanh"
        )

        self.fc = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        # rnn_out:
        # (batch, seq_len, hidden_size)

        rnn_out, hidden = self.rnn(x)

        logits = self.fc(rnn_out)

        return logits


model = DishRNN()

In [5]:
model

DishRNN(
  (rnn): RNN(5, 3, bias=False, batch_first=True)
  (fc): Linear(in_features=3, out_features=3, bias=False)
)

In [ ]:
# =====================================================
# 4. Training setup
# =====================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# =====================================================
# 5. Training loop
# =====================================================

epochs = 300

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    # reshape for CE loss
    loss = criterion(
        logits.view(-1, 3),
        y.view(-1)
    )

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {loss.item():.6f}"
        )

Epoch [50/300] Loss = 0.727067
Epoch [100/300] Loss = 0.407638
Epoch [150/300] Loss = 0.161681
Epoch [200/300] Loss = 0.090812
Epoch [250/300] Loss = 0.061349
Epoch [300/300] Loss = 0.045405


In [ ]:
# =====================================================
# 6. Evaluation
# =====================================================

with torch.no_grad():

    logits = model(X)

    preds = logits.argmax(dim=-1)

    accuracy = (
        (preds == y).float().mean().item()
    )

print("\nAccuracy:", accuracy)


Accuracy: 1.0


In [ ]:
# =====================================================
# 7. Test on all possible transitions
# =====================================================

print("\nLearned transitions:\n")

test_cases = [
    ('A', 'Sunny'),
    ('A', 'Rainy'),
    ('B', 'Sunny'),
    ('B', 'Rainy'),
    ('C', 'Sunny'),
    ('C', 'Rainy'),
]

hidden = None

for dish, weather in test_cases:

    x = encode_input(dish, weather)
    x = x.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(-1).item()

    print(
        f"Current Dish={dish:1s}, "
        f"Weather={weather:5s} "
        f"--> Predicted Next Dish={dishes[pred]}"
    )


Learned transitions:

Current Dish=A, Weather=Sunny --> Predicted Next Dish=A
Current Dish=A, Weather=Rainy --> Predicted Next Dish=B
Current Dish=B, Weather=Sunny --> Predicted Next Dish=B
Current Dish=B, Weather=Rainy --> Predicted Next Dish=C
Current Dish=C, Weather=Sunny --> Predicted Next Dish=B
Current Dish=C, Weather=Rainy --> Predicted Next Dish=A


In [ ]:
for name, param in model.named_parameters():
    print(name)
    print(param.data)
    print()

rnn.weight_ih_l0
tensor([[ 1.6995,  0.4584, -1.6581,  2.0460, -0.8809],
        [ 0.2785,  0.3362, -2.3999,  1.5364, -2.2622],
        [-1.8484,  1.8183, -1.0670,  1.0172, -2.3343]])

rnn.weight_hh_l0
tensor([[-0.8649,  1.4588, -1.5512],
        [ 0.2211, -0.0640, -0.2198],
        [ 1.6927, -1.5497,  1.5289]])

fc.weight
tensor([[-2.0026,  2.3299, -2.5996],
        [ 2.7051, -0.8554,  1.9113],
        [-1.4095, -0.9066,  2.7953]])



## Hidden State Update

$$
h_t
=
\tanh
\left(
W_{ih}x_t
+
W_{hh}h_{t-1}
+
b_{ih}
+
b_{hh}
\right)
$$

But, with `bias=False`, 
$$
\boxed{h_t = \tanh\left(W_{ih}x_t + W_{hh}h_{t-1}\right)}
$$

where,

$$
W_{ih} \in \mathbb{R}^{H \times D}
$$

$$
W_{hh} \in \mathbb{R}^{H \times H}
$$

$$
x_t \in \mathbb{R}^{D}
$$

$$
h_t \in \mathbb{R}^{H}
$$

For your model:

- $D = 5$ (A, B, C, Sunny, Rainy)
- $H = 3$

Thus

$$
W_{ih} \in \mathbb{R}^{3 \times 5}
$$

$$
W_{hh} \in \mathbb{R}^{3 \times 3}
$$

### Output Layer

$$
y_t
=
W_{fc}h_t
+
b_{fc}
$$

But, for `bias=False`, 

$$
\boxed{
y_t = W_{fc}h_t}
$$

where

$$
W_{fc} \in \mathbb{R}^{3 \times 3}
$$

and

$$
y_t \in \mathbb{R}^{3}
$$

The three output values are logits:

$$
y_t =
\begin{bmatrix}
s_A \\
s_B \\
s_C
\end{bmatrix}
$$

### Softmax

Convert logits into probabilities:

$$
p_i =
\frac{e^{s_i}}
{\sum_j e^{s_j}}
$$

or

$$
p(y_t = k)
=
\frac{e^{s_k}}
     {e^{s_A}+e^{s_B}+e^{s_C}}
$$

### Prediction

The predicted dish is

$$
\hat y_t
=
\arg\max_k p(y_t=k)
$$

which is equivalent to

$$
\hat y_t
=
\arg\max_k s_k
$$

because softmax preserves ordering.

### Explicit Form For Your Network

Input vector:

$$
x_t =
\begin{bmatrix}
A\\
B\\
C\\
Sunny\\
Rainy
\end{bmatrix}
$$

For example:

Dish = A, Weather = Rainy

$$
x_t =
\begin{bmatrix}
1\\
0\\
0\\
0\\
1
\end{bmatrix}
$$

### Hidden Unit Equations

Let

$$
h_t =
\begin{bmatrix}
h_{1,t}\\
h_{2,t}\\
h_{3,t}
\end{bmatrix}
$$

Then

$$
h_{1,t}
=
\tanh
\left(
w^{(1)}_{ih}x_t
+
w^{(1)}_{hh}h_{t-1}
\right)
$$

$$
h_{2,t}
=
\tanh
\left(
w^{(2)}_{ih}x_t
+
w^{(2)}_{hh}h_{t-1}
\right)
$$

$$
h_{3,t}
=
\tanh
\left(
w^{(3)}_{ih}x_t
+
w^{(3)}_{hh}h_{t-1}
\right)
$$

where each row of $W_{ih}$ and $W_{hh}$ corresponds to one hidden neuron.


### Output Logits

$$
s_A
=
w_A^\top h_t
$$

$$
s_B
=
w_B^\top h_t
$$

$$
s_C
=
w_C^\top h_t
$$

or

$$
\begin{bmatrix}
s_A\\
s_B\\
s_C
\end{bmatrix}
=
W_{fc}
\begin{bmatrix}
h_{1,t}\\
h_{2,t}\\
h_{3,t}
\end{bmatrix}
$$

### If Biases Were Enabled

PyTorch would compute

$$
h_t
=
\tanh
\left(
W_{ih}x_t
+
W_{hh}h_{t-1}
+
b_{ih}
+
b_{hh}
\right)
$$

and

$$
y_t
=
W_{fc}h_t
+
b_{fc}
$$

where

$$
b_{ih} \in \mathbb{R}^{H}
$$

$$
b_{hh} \in \mathbb{R}^{H}
$$

$$
b_{fc} \in \mathbb{R}^{3}
$$

For your current `bias=False` setup, all bias terms disappear.

### Discussion
In this laboratory, a Recurrent Neural Network (RNN) was developed and trained using PyTorch to understand how sequential data can be processed effectively. A dataset consisting of dish and weather combinations was generated and converted into numerical representations using one-hot encoding. The RNN model learned to predict the next dish based on the current dish and weather condition.

The most important feature of the RNN is its hidden state, which serves as a memory mechanism. This allows the network to retain relevant information from previous inputs and use it when making future predictions. During training, the model gradually reduced its prediction error and improved its accuracy by learning the transition patterns present in the sequence. The experiment highlighted the importance of recurrent connections in capturing temporal relationships and demonstrated how RNNs can successfully model sequence-dependent problems.

### Conclusion
This lab successfully demonstrated the implementation of a Recurrent Neural Network for sequence prediction tasks. By utilizing hidden states, the RNN was able to remember information from previous time steps and learn dependencies within the data. The trained model accurately predicted future states based on past observations, showing the effectiveness of recurrent architectures for handling sequential information. Overall, the experiment provided practical knowledge of RNN structure, training, and applications in areas such as natural language processing, speech recognition, and time-series forecasting.